# Mask coverage → pick a filtering threshold

The tiling pipeline (`preprocessing/tiling.py`) already computes a **per-tile coverage score** for
each mask type and stores it in the tiled dataset's `tiles.parquet` as these columns:
`tissue_overlap`, `epithelium_overlap`, `cancer_overlap`, `blur_overlap`, `folding_overlap`,
`residual_overlap`. Each is in `[0, 1]`: coverage columns are fractions of desired pixels and
`cancer_overlap` is the mean normalized cancer probability. Tiles can be kept/dropped by
thresholding these scores.

This notebook loads those columns straight from a tiled-dataset MLflow artifact and shows their
**distribution + CDF** so you can pick sensible thresholds.

> **Note on "AUC".** A true ROC/AUC needs a per-tile keep/discard ground truth, which we don't have.
> Instead we plot the **coverage distribution** and its **CDF** — the CDF axis directly answers
> *"if I set the threshold at T, what fraction of tiles do I keep vs drop?"*, which is exactly the
> decision you want to make. An optional luminal A/B overlay lets you check the two classes behave
> similarly under a given cutoff.

No masks or OpenSlide access needed — the overlaps are read as-is from the parquet, so the numbers
are exactly what the pipeline produced.

Edit the **Config** cell (tiled-dataset URI + which overlaps to look at), then run the **Calls** cells.

In [ ]:
!python -m ensurepip --upgrade
!python -m pip install "pandas" "mlflow>3" pyarrow matplotlib

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
from mlflow.artifacts import download_artifacts

# Config

This is the one cell you edit. Point `TILED_DATASET_URI` at the tiling run's MLflow artifact dir
(the one containing `slides.parquet` / `tiles.parquet`), and pick which overlaps to plot.

In [ ]:
MLFLOW_TRACKING_URI = "http://mlflow-s3.rationai-mlflow"
os.environ["MLFLOW_TRACKING_URI"] = MLFLOW_TRACKING_URI
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

# Tiled-dataset artifact dir (contains slides.parquet + tiles.parquet). Same style as the
# TILED_DATASETS entries in check_mask_coverage.ipynb. Point this at your dataset.
TILED_DATASET_URI = "mlflow-artifacts:/3/0b32eceb1c434abf94104bb2c54b792e/artifacts/mou_3_224"

# The overlap columns to analyze (as stored in tiles.parquet). Drop any you don't care about.
OVERLAP_COLUMNS = [
    "tissue_overlap",
    "epithelium_overlap",
    "cancer_overlap",
    "blur_overlap",
    "folding_overlap",
    "residual_overlap",
]

# All current overlap columns already use the notebook convention: higher means a larger
# fraction of the desired/keep signal. Only list legacy columns here if an older artifact stored
# them in the opposite direction.
INVERT_COLUMNS = []

# Column holding the luminal type on each tile row (for the optional A/B overlay). Set to None
# to disable the per-class split. tiling.py copies `type` onto every tile.
TYPE_COLUMN = "type"

# Functions

Run every cell in this section once, then use the **Calls** section at the bottom.

In [ ]:
# ---- Load tiles.parquet from a tiled-dataset MLflow artifact ----
# type_label mirrors ml/data/datasets/labels.py:_map_luminal_type  (a luminal -> 1, b luminal -> 0)
_LUMINAL_MAP = {"a luminal": 1, "b luminal": 0}


def load_tiles(uri=TILED_DATASET_URI, overlap_columns=OVERLAP_COLUMNS,
               invert_columns=INVERT_COLUMNS, type_column=TYPE_COLUMN):
    """Download the tiled dataset's tiles.parquet and return it as a DataFrame.

    Any column in `invert_columns` is replaced with `1 - value` (fixes overlaps stored with the
    opposite convention, so higher = cleaner/keep everywhere). Adds `type_label` (luminal A=1/B=0)
    if `type_column` is present. Warns about requested overlap columns missing from the parquet.
    """
    local_dir = Path(download_artifacts(uri))
    tiles = pd.read_parquet(local_dir / "tiles.parquet")
    print(f"Loaded {len(tiles):,} tiles from {local_dir / 'tiles.parquet'}")
    print(f"Columns: {tiles.columns.tolist()}")

    missing = [c for c in overlap_columns if c not in tiles.columns]
    if missing:
        print(f"WARNING: requested overlap columns not in parquet: {missing}")

    for col_name in invert_columns:
        if col_name in tiles.columns:
            tiles[col_name] = 1.0 - tiles[col_name]
            print(f"Inverted '{col_name}' -> plotting 1 - {col_name}")

    if type_column and type_column in tiles.columns:
        tiles["type_label"] = tiles[type_column].str.strip().str.lower().map(_LUMINAL_MAP)
        print(tiles[type_column].value_counts(dropna=False).to_string())
    return tiles

In [ ]:
# ---- Per-column quantile table (read cutoffs straight off it) ----
def summary_table(tiles, overlap_columns=OVERLAP_COLUMNS):
    """Count + quantiles of each overlap column.

    Includes fine LOW-end quantiles (1/2/5/10%) because for right-skewed overlaps the useful
    threshold sits in the bottom tail: e.g. the 5% row tells you the cutoff that drops ~5% of tiles.
    """
    qs = [0.0, 0.01, 0.02, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 1.0]
    cols = [c for c in overlap_columns if c in tiles.columns]
    out = (
        tiles[cols]
        .describe(percentiles=qs)
        .drop(index=["mean", "std"], errors="ignore")
        .T
    )
    return out

In [ ]:
# ---- Plot: coverage histogram + CDF, with candidate thresholds ----
_CLASS_NAMES = {1: "a luminal", 0: "b luminal"}


def auto_xlim(values, tail=0.02, pad=0.01):
    """Zoom window that frames the informative tail of a skewed distribution.

    Returns (lo, hi) spanning the `tail` quantile (e.g. 2%) up to the max, padded a little. When
    almost every tile has coverage ~1.0, this zooms in near 1.0 where the threshold decision lives.
    Falls back to the full (0, 1) range for empty/all-NaN or degenerate (all-identical) input.
    """
    values = np.asarray(pd.Series(values).dropna())
    if values.size == 0:
        return (0.0, 1.0)
    lo = float(np.quantile(values, tail))
    hi = float(values.max())
    lo = max(0.0, lo - pad)
    hi = min(1.0, hi + pad)
    if hi - lo < 1e-6:  # degenerate (all identical) -> fall back to full range
        return (0.0, 1.0)
    return (lo, hi)


def plot_overlap(tiles, column, thresholds=(0.1, 0.25, 0.5), by_class=True,
                 bins=50, xlim=(0.0, 1.0)):
    """Histogram (left) + CDF (right) of one overlap column.

    The CDF reads directly as 'fraction of tiles dropped if threshold = x'; keep-fraction is
    printed for each candidate threshold. Optional luminal A/B overlay.

    `xlim` sets the x-axis window (bins + histogram range track it), so for heavily skewed
    columns you can zoom toward 1.0. Pass `xlim=auto_xlim(tiles[column].dropna())` to auto-frame
    the tail. The keep/drop stats below always use the FULL data, regardless of the zoom window.
    """
    if column not in tiles.columns:
        print(f"Column '{column}' not in tiles.")
        return
    values_all = tiles[column].dropna()
    if values_all.empty:
        print(f"No values for '{column}' (empty or all-NaN) -- skipping.")
        return

    lo, hi = xlim
    fig, (ax_hist, ax_cdf) = plt.subplots(1, 2, figsize=(13, 4.5))

    def _plot_series(values, label):
        values = np.asarray(values)
        ax_hist.hist(values, bins=bins, range=(lo, hi), alpha=0.5, density=True, label=label)
        xs = np.sort(values)
        ys = np.arange(1, len(xs) + 1) / len(xs)  # P(overlap <= x) = fraction dropped
        ax_cdf.plot(xs, ys, label=label)

    if by_class and "type_label" in tiles.columns and tiles["type_label"].notna().any():
        for lbl, group in tiles.dropna(subset=[column]).groupby("type_label", dropna=True):
            _plot_series(group[column], _CLASS_NAMES.get(lbl, str(lbl)))
    else:
        _plot_series(values_all, "all")

    for t in thresholds:
        if lo <= t <= hi:
            ax_hist.axvline(t, color="k", ls="--", lw=0.8)
            ax_cdf.axvline(t, color="k", ls="--", lw=0.8)

    zoom_note = "" if (lo, hi) == (0.0, 1.0) else f"  [zoom {lo:.3f}–{hi:.3f}]"
    ax_hist.set(title=f"{column}: distribution{zoom_note}", xlabel="coverage", ylabel="density",
                xlim=(lo, hi))
    ax_hist.legend()
    ax_cdf.set(title=f"{column}: CDF (= fraction dropped at threshold){zoom_note}",
               xlabel="threshold", ylabel="fraction of tiles with coverage ≤ threshold",
               xlim=(lo, hi))
    ax_cdf.legend()
    fig.tight_layout()
    plt.show()

    total = len(values_all)
    print(f"{column}: {total:,} tiles")
    for t in thresholds:
        kept = int((values_all > t).sum())
        print(f"  threshold > {t:>5}: keep {kept:>9,} ({100 * kept / total:5.1f}%), "
              f"drop {total - kept:>9,} ({100 * (total - kept) / total:5.1f}%)")

# Calls

Run these to load the dataset and inspect each overlap. Adjust `thresholds=` to your candidate
cutoffs and re-run cell 3 as needed.

In [ ]:
# 1. Load tiles.parquet from the tiled-dataset artifact.
tiles = load_tiles()
tiles.head()

In [ ]:
# 2. Quantile table — read candidate thresholds straight off it.
summary_table(tiles)

In [ ]:
# 3. Distribution + CDF per overlap column.
#    - `thresholds=` : candidate cutoffs to annotate + print keep/drop for.
#    - `xlim=`       : x-axis zoom window. Only folding/residual are zoomed here (their mass sits
#                      near 1.0); the rest are shown full-range. auto_xlim frames the tail --
#                      lower `tail=` to zoom tighter, or pass a manual window like (0.9, 1.0).
ZOOM_COLUMNS = ["folding_overlap", "residual_overlap"]

for column in OVERLAP_COLUMNS:
    if column not in tiles.columns:
        continue
    if column in ZOOM_COLUMNS:
        xlim = auto_xlim(tiles[column].dropna())  # or e.g. (0.9, 1.0)
    else:
        xlim = (0.0, 1.0)
    plot_overlap(tiles, column, thresholds=(0.1, 0.25, 0.5), xlim=xlim)

## How to pick a threshold

There's no single "correct" cutoff — you're trading off **tile quality** against **how much data you keep**. Use the plots + quantile table together:

1. **Look for a gap / elbow in the distribution.** If the histogram is bimodal — a big clean lump near 1.0 and a smaller junk lump near 0 — put the threshold *in the valley between them*. That's the most defensible cutoff: it separates two populations rather than cutting into one.
2. **If there's no clear gap (just a smooth skew toward 1.0),** there's no natural boundary, so pick by **data-loss budget** instead. Decide how many tiles you're willing to drop (e.g. "≤5%") and read the cutoff off the matching low-end quantile in `summary_table` (the 5% row) or off the CDF (find where y = 0.05, take the x). A skewed column usually means the mask rarely fires, so a low cutoff (drop only the worst 1–5%) is typically enough.
3. **Match the cutoff to what the mask is *for*.**
   - `tissue_overlap` — the gate that decides a tile is on tissue at all; a moderate cutoff (e.g. 0.25–0.5) is normal. (The pipeline already drops `tissue_overlap == 0`.)
   - `epithelium_overlap` — how much tumor epithelium a tile must contain; set by how epithelium-pure you want tiles, not by data loss.
   - `blur` / `folding` / `residual` (QC) — these only remove artifacts, so keep them **lenient**: drop just the clearly-bad tail (often 0.01–0.1) so you don't throw away good tissue.
4. **Sanity-check the A/B overlay.** A QC/tissue threshold should affect luminal A and B tiles *about equally*. If one class loses far more tiles than the other at your cutoff, you're introducing class bias — loosen it or reconsider.
5. **Confirm with the printed keep/drop line.** Once a candidate looks right, add it to `thresholds=` and check the exact keep/drop counts are acceptable before committing.

In short: **cut at a gap if one exists; otherwise cut by an acceptable data-loss budget — lenient for pure-artifact masks, and balanced across A/B.**